# Engram against a plain .md memory file (LongMemEval)

Gemma 3 4B writes a short note after each conversation; the notes form one memory file, like the memory in everyday chat apps. Then it answers the test questions from that file. Engram's answers to the same questions are already recorded. Rule fixed in `frozen_md_baseline.json`. Benchmark data only.

**Kaggle:** Settings > Accelerator > GPU T4 x2, and Settings > Turn on internet. Then Save Version > Save & Run All. About 6 hours (the notes are limited to 5 hours). If it stops early, run it again: it continues where it stopped.


In [ ]:
COMMIT = "8f0bd36"
import os, subprocess, sys, time, urllib.request
gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
if not gpu: raise SystemExit("No GPU. Choose Settings > Accelerator > GPU T4 x2.")
print("GPU:", gpu.replace(chr(10), ", "))
try:
    urllib.request.urlopen("https://github.com", timeout=10)
except Exception:
    raise SystemExit("No internet. Choose Settings > Turn on internet, then run again.")
WORK, SAVE = "/kaggle/working", "/kaggle/working/engram_results"
os.makedirs(SAVE, exist_ok=True); os.chdir(WORK)
subprocess.run("rm -rf Engram; git clone -q https://github.com/manjunathhari/Engram.git", shell=True, check=True)
subprocess.run(f"cd Engram && git checkout -q {COMMIT}", shell=True, check=True)
DATA = f"{WORK}/longmemeval_s_cleaned.json"
if not os.path.exists(DATA):
    subprocess.run(f"wget -q -O {DATA} https://huggingface.co/datasets/xiaowu0162/longmemeval-cleaned/resolve/main/longmemeval_s_cleaned.json", shell=True, check=True)
print("data:", os.path.getsize(DATA) // 1000000, "MB")
subprocess.run("apt-get -qq install -y zstd pciutils > /dev/null; curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1", shell=True)
env = dict(os.environ, OLLAMA_CONTEXT_LENGTH="4096", OLLAMA_KEEP_ALIVE="-1", OLLAMA_NUM_PARALLEL="4")
subprocess.Popen(["ollama", "serve"], env=env, stdout=open(f"{WORK}/ollama.log", "a"), stderr=subprocess.STDOUT, start_new_session=True)
for _ in range(60):
    try: urllib.request.urlopen("http://localhost:11434/api/version", timeout=2); break
    except Exception: time.sleep(2)
else: raise SystemExit("Ollama did not start. See ollama.log in the Output.")
MODEL = "hf.co/bartowski/google_gemma-3-4b-it-GGUF:Q4_K_M"
r = subprocess.run(["ollama", "pull", MODEL], capture_output=True, text=True)
if r.returncode: raise SystemExit("Model download failed: " + r.stderr[-200:])
subprocess.run(["ollama", "cp", MODEL, "gemma3:4b"])
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)
LOG = f"{SAVE}/md_run.log"
p = subprocess.Popen([sys.executable, "-u", "kit_tools/md_test.py", "--data", DATA, "--url", "http://localhost:11434/v1", "--model", "gemma3:4b", "--save", SAVE],
                     cwd=f"{WORK}/Engram", stdout=open(LOG, "a"), stderr=subprocess.STDOUT)
while p.poll() is None:
    time.sleep(60)
    t = open(LOG).read().splitlines()
    print(chr(13) + " | ".join(t[-2:])[-160:], end="")
print(); print(chr(10).join(open(LOG).read().splitlines()[-25:]))
if "Done" not in open(LOG).read(): raise SystemExit("The run did not finish. Run again to continue, or send Claude md_run.log.")


In [ ]:
os.chdir("/kaggle/working")
from IPython.display import FileLink, display
for f in ("md_answers.jsonl", "md_plan.json", "md_notes.json", "md_run.log"):
    display(FileLink(f"engram_results/{f}"))


Send Claude `md_answers.jsonl`, `md_plan.json`, `md_notes.json` and `md_run.log`.
